In [13]:
import pandas as pd
from sentence_transformers import SentenceTransformer
import numpy as np


In [3]:
notes = pd.read_csv("../data/raw/clinical_notes.csv")

print("Original notes:", len(notes))
print(notes.columns.tolist())

Original notes: 1602
['ingest_timestamp', 'clinical_note_id', 'clean_note_text', 'creation_timestamp', 'updt_dt_tm', 'note_subject', 'note_type', 'admission_id', 'person_id']


In [4]:
# Remove broken notes
notes_clean = notes[
    notes["clean_note_text"].astype(str).str.strip() != "#NAME?"
].copy()

print("After #NAME? removal:", len(notes_clean))

After #NAME? removal: 1595


In [5]:
# Deduplicate repeated note content per patient
notes_dedup = (
    notes_clean
    .sort_values(["person_id", "creation_timestamp"])
    .drop_duplicates(
        subset=["person_id", "clean_note_text"],
        keep="first"
    )
    .reset_index(drop=True)
)

print("After deduplication:", len(notes_dedup))
print("Patients:", notes_dedup["person_id"].nunique())

After deduplication: 1103
Patients: 50


In [6]:
print(notes_dedup.columns.tolist())

['ingest_timestamp', 'clinical_note_id', 'clean_note_text', 'creation_timestamp', 'updt_dt_tm', 'note_subject', 'note_type', 'admission_id', 'person_id']


In [7]:
# Create a new dataframe with only the relevant columns for chunking

whole_chunks = (
    notes_dedup[
        [
            "person_id",
            "creation_timestamp",
            "clean_note_text"
        ]
    ]
    .copy()
    .rename(columns={
        "clean_note_text": "chunk_text"
    })
)

# Unique identifier for every chunk/note
whole_chunks["chunk_id"] = range(len(whole_chunks))

# Ensure chronological ordering within each patient
whole_chunks = (
    whole_chunks
    .sort_values(
        ["person_id", "creation_timestamp"]
    )
    .reset_index(drop=True)
)

print("Total chunks:", len(whole_chunks))
print("Patients:", whole_chunks["person_id"].nunique())

display(whole_chunks.head())

Total chunks: 1103
Patients: 50


,person_id,creation_timestamp,chunk_text,chunk_id
0,028998ee-babc-4096-9b28-001bc2f9a84e,07/01/2026 08:45,"- Patient: Tomos Ellis, 15-year-old male, pre...",0
1,028998ee-babc-4096-9b28-001bc2f9a84e,07/01/2026 09:10,"Patient: Tomos Ellis, 15-year-old male, presen...",1
2,028998ee-babc-4096-9b28-001bc2f9a84e,07/01/2026 09:25,"Patient name: Tomos Ellis, 15-year-old male. N...",2
3,028998ee-babc-4096-9b28-001bc2f9a84e,07/01/2026 10:00,"Reviewed abdominal X-ray on 2026-01-07, which ...",3
4,028998ee-babc-4096-9b28-001bc2f9a84e,07/01/2026 10:30,Patient\nTomos Ellis\n\nAge\n15\n\nSex\nMale\n...,4


In [9]:
# Load the BGE model for embeddings

bge_model = SentenceTransformer(
    "BAAI/bge-base-en-v1.5"
)

print("BGE model loaded.")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BGE model loaded.


In [10]:
# Create embeddings for all chunks

chunk_texts = whole_chunks["chunk_text"].astype(str).tolist()

chunk_embeddings = bge_model.encode(
    chunk_texts,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True
)

print("Embedding shape:", chunk_embeddings.shape)

Batches:   0%|          | 0/35 [00:00<?, ?it/s]

Embedding shape: (1103, 768)


In [17]:
RAG_QUERY = """
Retrieve the clinically relevant information needed to produce a
comprehensive longitudinal summary of this patient's clinical history,
including major diagnoses, treatments, investigations, clinical
progression, and outcomes.
""".strip()


def retrieve_patient_notes(
    person_id,
    whole_chunks,
    chunk_embeddings,
    query_text=RAG_QUERY,
    top_k=20
):
    """
    Retrieve the top-k most relevant whole-note chunks
    for one patient using BGE similarity.
    """

    # Find this patient's rows in whole_chunks
    patient_indices = np.where(
        whole_chunks["person_id"].to_numpy() == person_id
    )[0]

    # Patient-specific embeddings
    patient_embeddings = chunk_embeddings[patient_indices]

    # Encode query with the same BGE model/settings
    query_embedding = bge_model.encode(
        query_text,
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=False
    )

    # Dot product = cosine similarity because embeddings are normalized
    scores = np.dot(
        patient_embeddings,
        query_embedding
    )

    # In case a patient has fewer than 20 notes
    k = min(top_k, len(patient_indices))

    # Top-k within THIS patient only
    local_top_indices = np.argsort(scores)[-k:][::-1]

    # Convert back to global dataframe indices
    global_top_indices = patient_indices[local_top_indices]

    # Keep metadata + similarity score
    retrieved = whole_chunks.iloc[global_top_indices].copy()

    retrieved["similarity_score"] = scores[local_top_indices]

    return retrieved

In [18]:
test_patient = whole_chunks["person_id"].iloc[0]

retrieved = retrieve_patient_notes(
    person_id=test_patient,
    whole_chunks=whole_chunks,
    chunk_embeddings=chunk_embeddings
)

print("Patient:", test_patient)
print("Retrieved notes:", len(retrieved))

display(
    retrieved[
        [
            "chunk_id",
            "person_id",
            "creation_timestamp",
            "similarity_score"
        ]
    ]
)

Patient: 028998ee-babc-4096-9b28-001bc2f9a84e
Retrieved notes: 15


,chunk_id,person_id,creation_timestamp,similarity_score
0,0,028998ee-babc-4096-9b28-001bc2f9a84e,07/01/2026 08:45,0.542850
8,8,028998ee-babc-4096-9b28-001bc2f9a84e,07/01/2026 16:00,0.542014
10,10,028998ee-babc-4096-9b28-001bc2f9a84e,08/01/2026 10:30,0.525251
4,4,028998ee-babc-4096-9b28-001bc2f9a84e,07/01/2026 10:30,0.523271
2,2,028998ee-babc-4096-9b28-001bc2f9a84e,07/01/2026 09:25,0.519711
11,11,028998ee-babc-4096-9b28-001bc2f9a84e,08/01/2026 11:15,0.517823
7,7,028998ee-babc-4096-9b28-001bc2f9a84e,07/01/2026 15:30,0.515500
5,5,028998ee-babc-4096-9b28-001bc2f9a84e,07/01/2026 11:00,0.515225
12,12,028998ee-babc-4096-9b28-001bc2f9a84e,08/01/2026 14:00,0.513148
6,6,028998ee-babc-4096-9b28-001bc2f9a84e,07/01/2026 13:00,0.506596


In [19]:
# Ensure chronological ordering of retrieved notes

retrieved["creation_timestamp"] = pd.to_datetime(
    retrieved["creation_timestamp"],
    dayfirst=True
)

retrieved_chronological = (
    retrieved
    .sort_values("creation_timestamp")
    .reset_index(drop=True)
)

display(
    retrieved_chronological[
        [
            "chunk_id",
            "creation_timestamp",
            "similarity_score"
        ]
    ]
)

,chunk_id,creation_timestamp,similarity_score
0,0,2026-01-07 08:45:00,0.542850
1,1,2026-01-07 09:10:00,0.493814
2,2,2026-01-07 09:25:00,0.519711
3,3,2026-01-07 10:00:00,0.491517
4,4,2026-01-07 10:30:00,0.523271
5,5,2026-01-07 11:00:00,0.515225
6,6,2026-01-07 13:00:00,0.506596
7,7,2026-01-07 15:30:00,0.515500
8,8,2026-01-07 16:00:00,0.542014
9,9,2026-01-08 09:00:00,0.487196


In [20]:
# Build RAG context from retrieved notes

def build_rag_context(retrieved_chronological):
    context_parts = []

    for i, row in retrieved_chronological.iterrows():

        timestamp = row["creation_timestamp"].strftime(
            "%Y-%m-%d %H:%M"
        )

        context_parts.append(
            f"""[SOURCE NOTE {i + 1}]
Creation timestamp: {timestamp}

{row["chunk_text"]}"""
        )

    return "\n\n---\n\n".join(context_parts)


rag_context = build_rag_context(
    retrieved_chronological
)

print(rag_context[:5000])

[SOURCE NOTE 1]
Creation timestamp: 2026-01-07 08:45

 - Patient: Tomos Ellis, 15-year-old male, presented via A&E on 07/01/26 at 08:45 with severe abdominal pain over the past 3 days. - Date of birth: 2008-05-17. - NHS number: 965833270. - Triage category: 3 - Urgent. - No known allergies reported. - Current medications: None declared.- Past medical history: None documented. - Initial assessment performed by Nurse Jamie Leigh Alexander. - ED diagnosis: Constipation. - Admiting consultant: Dr. Susan Jennifer Robson. - Decision: Proceed with baseline obs and assess pain severity.
Nurse Jamie Leigh Alexander 
NMC number: 27H5222T

---

[SOURCE NOTE 2]
Creation timestamp: 2026-01-07 09:10

Patient: Tomos Ellis, 15-year-old male, presenting with abdominal pain.

- Event date/time: 07/01/26 at 09:10.
- Performed focused abdominal assessment by Nurse Jamie Leigh Alexander.
- Findings: Mild abdo distension, tenderness in lower abdo.

- NO (No guarding, rebound tenderness).
- Advise to NPO.
- 

In [22]:
import config.prompts as prompts_module
from src.llm.llm import generate_summary

summary = generate_summary(
    context=rag_context,
    prompt=prompts_module.SUMMARY_PROMPT
)

ModuleNotFoundError: No module named 'config'